In [1]:
# ==== 1. Import Library ====
import pandas as pd
import numpy as np
import math

In [2]:
# ==== 2. Load Data ====
# Ganti path sesuai lokasi file CSV kamu
df = pd.read_csv("Data Clean.csv")

print(f"Total data: {len(df)} baris")
print(f"Total brand: {df['brand_name'].nunique()} restoran")
df.head()

Total data: 8061 baris
Total brand: 25 restoran


,brand_name,branch_name,rating,review_text,review_date,review_date_converted,year,sentiment,month_name,review_text_id,review_text_clean,review_text_id_clean,review_text_clean_lower,review_text_id_clean_lower
0,Unagi Kurofune Indonesia,Unagi Kurofune Indonesia,5,High quality food meets high quality service.\...,Edited a month ago,2025-12-20,2025,positive,Desember,Makanan berkualitas tinggi bertemu dengan laya...,High quality food meets high quality service T...,Makanan berkualitas tinggi bertemu dengan laya...,high quality food meets high quality service t...,makanan berkualitas tinggi bertemu dengan laya...
1,Unagi Kurofune Indonesia,Unagi Kurofune Indonesia,5,2 months reservation wow insane level of queue...,2 weeks ago,2026-01-05,2026,positive,Januari,Reservasi 2 bulan wow tingkat antrian yang gil...,months reservation wow insane level of queue p...,Reservasi bulan wow tingkat antrian yang gila ...,months reservation wow insane level of queue p...,reservasi bulan wow tingkat antrian yang gila ...
2,Unagi Kurofune Indonesia,Unagi Kurofune Indonesia,5,The viral unagi spot in South Jakarta!\nFinall...,6 days ago,2026-01-13,2026,positive,Januari,Spot Unagi yang Viral di Jaksel!\nAkhirnya men...,The viral unagi spot in South Jakarta Finally ...,Spot Unagi yang Viral di Jaksel Akhirnya menco...,the viral unagi spot in south jakarta finally ...,spot unagi yang viral di jaksel akhirnya menco...
3,Unagi Kurofune Indonesia,Unagi Kurofune Indonesia,5,Decided to give this most wanted unagi in town...,Edited 2 months ago,2025-11-20,2025,positive,November,Memutuskan untuk memberikan unagi yang paling ...,Decided to give this most wanted unagi in town...,Memutuskan untuk memberikan unagi yang paling ...,decided to give this most wanted unagi in town...,memutuskan untuk memberikan unagi yang paling ...
4,Unagi Kurofune Indonesia,Unagi Kurofune Indonesia,5,came at 11.15 before the restaurant open and g...,a month ago,2025-12-20,2025,positive,Desember,datang jam 11.15 sebelum restoran buka dan men...,came at before the restaurant open and got the...,datang jam sebelum restoran buka dan mendapat ...,came at before the restaurant open and got the...,datang jam sebelum restoran buka dan mendapat ...


In [3]:
# ==== 3. Tampilkan Daftar Restoran yang Tersedia ====
daftar_brand = sorted(df['brand_name'].unique().tolist())
print("Daftar restoran yang tersedia:")
print("-" * 40)
for i, brand in enumerate(daftar_brand, 1):
    print(f"{i:2}. {brand}")

Daftar restoran yang tersedia:
----------------------------------------
 1. Bobby’s Burgers
 2. Chikuro
 3. Dream Dates Artisan Bakery & Restaurant
 4. Five Monkeys Burger
 5. Gabon African
 6. Iga Panggang Panglima BBQ Ribs
 7. Kappa Sushi
 8. Lawless Burgerbar
 9. Little Olie
10. Little Red Dot
11. Meatguy Steakhouse SCBD
12. Mie Gacoan
13. Mil Toast House
14. Nasi Goreng Kambing Kebon Sirih
15. Obihiro Nikudon
16. Osteria Gia
17. Paris Baguette
18. Potteria
19. Sate Taichan Bang Yoyo
20. Sec Bowl
21. Sushi Maru
22. Sànùk Thai Boat Noodle
23. UNION
24. Unagi Kurofune Indonesia
25. 海底捞火锅 Haidilao Hot Pot


In [4]:
# ==== 4. Preprocessing Semua Data ====

# Pastikan kolom sentiment tidak ada NaN
df['sentiment'] = df['sentiment'].fillna('neutral')

# Mapping sentiment ke skor 0–1 (case-insensitive)
sentiment_mapping = {
    "positive": 1.0,
    "neutral":  0.5,
    "negative": 0.0
}

df['sentiment_score'] = df['sentiment'].str.lower().map(sentiment_mapping)
# Kalau ada label lain yang aneh, fallback ke 0.5 (netral)
df['sentiment_score'] = df['sentiment_score'].fillna(0.5)

# Rating 1–5 diubah ke 0–1
df['rating_score'] = df['rating'] / 5.0

print("Preprocessing selesai.")

Preprocessing selesai.


In [5]:
# ==== 5. Agregasi per Brand & Branch ====
group_cols = ['brand_name', 'branch_name']

agg = (
    df.groupby(group_cols)
      .agg(
          avg_rating    = ('rating', 'mean'),
          rating_score  = ('rating_score', 'mean'),
          avg_sentiment = ('sentiment_score', 'mean'),
          n_reviews     = ('rating', 'count')
      )
      .reset_index()
)

print(f"Total cabang yang diproses: {len(agg)}")
agg.head()

Total cabang yang diproses: 107


,brand_name,branch_name,avg_rating,rating_score,avg_sentiment,n_reviews
0,Bobby’s Burgers,Bobby's Burger - PIM 3,4.823529,0.964706,0.953782,119
1,Bobby’s Burgers,Bobby's Burgers - Lippo Mall Puri,4.857143,0.971429,0.975155,161
2,Bobby’s Burgers,Bobby’s Burgers - ASHTA,4.304348,0.860870,0.844720,161
3,Chikuro,Chikuro Gandaria City,3.779661,0.755932,0.703390,236
4,Chikuro,Chikuro Kuningan City,3.189189,0.637838,0.554054,37


In [6]:
# ==== 6. Volume Score ====
# Kita anggap 100 review = nilai penuh (1.0)
MAX_REVIEWS = 100

agg['volume_score'] = np.clip(agg['n_reviews'] / MAX_REVIEWS, 0, 1)

In [7]:
# ==== 7. Hitung Final Score ====
# Bobot (bisa diubah, tapi total harus = 1)
w_rating    = 0.3   # 30% dari rating
w_sentiment = 0.3   # 30% dari sentimen
w_volume    = 0.4   # 40% dari jumlah review

# Cek jumlah bobot = 1
total_weight = w_rating + w_sentiment + w_volume
print("Total weight:", total_weight)
if not math.isclose(total_weight, 1.0):
    raise ValueError("Bobot tidak berjumlah 1. Tolong sesuaikan lagi.")

# Hitung skor akhir
agg['final_score'] = (
    w_rating    * agg['rating_score'] +
    w_sentiment * agg['avg_sentiment'] +
    w_volume    * agg['volume_score']
)

# ==== 8. Kategorisasi per Cabang ====
THRESHOLD = 0.9

agg['kategori_makanan'] = np.where(
    agg['final_score'] >= THRESHOLD,
    "Enak",
    "Kurang / FOMO"
)

# Urutkan dari skor tertinggi
hasil = agg.sort_values('final_score', ascending=False).reset_index(drop=True)
print("Kalkulasi selesai untuk semua restoran.")

Total weight: 1.0
Kalkulasi selesai untuk semua restoran.


In [8]:
# ==== 9. Rekap per Brand (semua cabang digabung) ====
brand_summary = (
    hasil
    .groupby('brand_name')
    .apply(lambda g: pd.Series({
        'total_cabang'         : len(g),
        'total_reviews'        : g['n_reviews'].sum(),
        'weighted_final_score' : (g['final_score'] * g['n_reviews']).sum() / g['n_reviews'].sum(),
        'jumlah_cabang_enak'   : (g['kategori_makanan'] == 'Enak').sum(),
        'jumlah_cabang_fomo'   : (g['kategori_makanan'] != 'Enak').sum()
    }))
    .reset_index()
)

# Tentukan kategori brand: Enak atau Kurang / FOMO
brand_summary['kategori_brand'] = np.where(
    brand_summary['weighted_final_score'] >= THRESHOLD,
    'Enak',
    'Kurang / FOMO'
)

brand_summary = brand_summary.sort_values('weighted_final_score', ascending=False).reset_index(drop=True)
print(f"Brand summary siap untuk {len(brand_summary)} restoran.")

Brand summary siap untuk 25 restoran.


/var/folders/mj/sx1wrw_j7c1fmv27z8h81_sr0000gn/T/ipykernel_6766/2071981946.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [9]:
# ==== 10. USER INPUT: Cari Restoran ====

def cari_restoran(nama_input):
    """
    Cari restoran berdasarkan nama (tidak case-sensitive, partial match).
    Tampilkan hasil kategorisasi lengkap.
    """
    nama_lower = nama_input.strip().lower()
    
    # Cari brand yang cocok (partial match)
    cocok = brand_summary[
        brand_summary['brand_name'].str.lower().str.contains(nama_lower, na=False)
    ]
    
    if cocok.empty:
        print(f"\n❌ Restoran '{nama_input}' tidak ditemukan dalam data.")
        print("\nTips: Coba ketik sebagian nama, contoh: 'mie', 'sushi', 'burger'")
        print("\nDaftar brand yang tersedia:")
        for b in sorted(brand_summary['brand_name'].tolist()):
            print(f"  - {b}")
        return
    
    for _, row in cocok.iterrows():
        brand = row['brand_name']
        print("\n" + "=" * 55)
        print(f"  🍽️  {brand}")
        print("=" * 55)
        print(f"  Total cabang          : {int(row['total_cabang'])}")
        print(f"  Total review          : {int(row['total_reviews'])}")
        print(f"  Weighted final score  : {row['weighted_final_score']:.3f}")
        print(f"  Cabang Enak           : {int(row['jumlah_cabang_enak'])}")
        print(f"  Cabang Kurang/FOMO    : {int(row['jumlah_cabang_fomo'])}")
        
        # Tampilkan emoji sesuai kategori
        if row['kategori_brand'] == 'Enak':
            label = "✅  ENAK"
        else:
            label = "⚠️   KURANG / FOMO"
        
        print(f"\n  Kesimpulan brand      : {label}")
        print("=" * 55)
        
        # Detail per cabang (kalau lebih dari 1)
        cabang_brand = hasil[hasil['brand_name'] == brand][[
            'branch_name', 'avg_rating', 'avg_sentiment',
            'n_reviews', 'final_score', 'kategori_makanan'
        ]].reset_index(drop=True)
        
        if len(cabang_brand) > 1:
            print(f"\n  Detail per cabang ({len(cabang_brand)} cabang):")
            print(cabang_brand.to_string(index=False))
        print()


# ---- Jalankan di sini ----
# Ganti nama restoran sesuai yang ingin dicari
nama_restoran = input("Masukkan nama restoran yang ingin dicari: ")
cari_restoran(nama_restoran)


  🍽️  Mie Gacoan
  Total cabang          : 26
  Total review          : 424
  Weighted final score  : 0.636
  Cabang Enak           : 0
  Cabang Kurang/FOMO    : 26

  Kesimpulan brand      : ⚠️   KURANG / FOMO

  Detail per cabang (26 cabang):
                                  branch_name  avg_rating  avg_sentiment  n_reviews  final_score kategori_makanan
             Mie Gacoan Jakarta - Pulo Gebang    5.000000       1.000000         20     0.680000    Kurang / FOMO
           Mie Gacoan Jakarta - Lenteng Agung    5.000000       1.000000         20     0.680000    Kurang / FOMO
              Mie Gacoan Jakarta - Peta Utara    4.950000       1.000000         20     0.677000    Kurang / FOMO
           Mie Gacoan Jakarta - Gunung Sahari    4.944444       1.000000         18     0.668667    Kurang / FOMO
               Mie Gacoan Jakarta - Fatmawati    4.944444       1.000000         18     0.668667    Kurang / FOMO
             Mie Gacoan Jakarta - Radin Inten    4.941176       1.0000

In [10]:
# ==== BONUS: Tampilkan Semua Hasil Kategorisasi ====
# (Opsional, jalankan kalau mau lihat semua sekaligus)

print("REKAP SEMUA RESTORAN")
print("=" * 65)
tampil = brand_summary[['brand_name', 'total_cabang', 'total_reviews',
                         'weighted_final_score', 'kategori_brand']].copy()
tampil.columns = ['Brand', 'Cabang', 'Reviews', 'Score', 'Kategori']
print(tampil.to_string(index=False))

REKAP SEMUA RESTORAN
                                  Brand  Cabang  Reviews    Score      Kategori
Dream Dates Artisan Bakery & Restaurant     2.0    425.0 0.987224          Enak
                             Sushi Maru     2.0    407.0 0.985553          Enak
               Unagi Kurofune Indonesia     1.0    495.0 0.985091          Enak
                        Mil Toast House     1.0    500.0 0.983020          Enak
                          Gabon African     1.0    434.0 0.968548          Enak
                        Obihiro Nikudon     1.0    500.0 0.966580          Enak
                Meatguy Steakhouse SCBD     1.0    260.0 0.965962          Enak
                               Potteria     1.0    300.0 0.959000          Enak
                        Bobby’s Burgers     3.0    441.0 0.955306          Enak
                            Little Olie     1.0    493.0 0.953022          Enak
                            Kappa Sushi     4.0    385.0 0.951179          Enak
                   